In [1]:
# Librerias necesarias
import os
import sys
import json
import time
import numpy as np
import pandas as pd
from copy import deepcopy
from generacion_pacientes import generar_pacientes
from collections import deque
# Add the parent directory to sys.path to allow relative imports
sys.path.append(os.path.abspath(os.path.join('..', '1. codigo analisis')))
import parametros as p
from kpis import * # Mala practica pero facil
from clases import Paciente, Simulacion, ModeloA

In [10]:
def correr_simulaciones_sensibilidad_simple(
    clase_modelo,
    T_max=4500,
    ciclos=4208,
    num_simulaciones=5,
    seed_inicial=0,
    pacientes_caso_base=False,
    log_detallado=True,
    carpeta_resultado="resultados sensibilidad",
    info_cambio=None  # dict: {"unidad": "ICU", "hospital": 2, "delta": +1}
):
    import parametros as p  # Asegúrate de tenerlo disponible

    nombre_modelo = clase_modelo.__name__
    nombre_carpeta = f"{nombre_modelo}_T{T_max}_C{ciclos}"

    if info_cambio:
        delta = info_cambio["delta"]
        unidad = info_cambio["unidad"]#.replace("/", "_")
        hospital = info_cambio["hospital"]
        sufijo = f"_H{hospital}_{unidad}_{'+' if delta >= 0 else ''}{delta}".replace("/", "_")
        nombre_carpeta += sufijo

    base_dir = os.path.join(carpeta_resultado, nombre_carpeta)
    logs_dir = os.path.join(base_dir, "logs")
    plots_dir = os.path.join(base_dir, "plots")
    kpis_dir = os.path.join(base_dir, "kpis")
    os.makedirs(logs_dir, exist_ok=True)
    os.makedirs(plots_dir, exist_ok=True)
    os.makedirs(kpis_dir, exist_ok=True)

    # Aplicar cambio temporal a parametros.py
    if info_cambio:
        hospital_id = info_cambio["hospital"]
        unidad_nombre = info_cambio["unidad"]
        delta = info_cambio["delta"]
        id_unidad = p.dict_unidades[unidad_nombre]

        original = p.dict_capacidades[hospital_id][id_unidad]
        nueva = original + delta

        if nueva < 1:
            print(f"❌ Capacidad inválida (<1): H{hospital_id} {unidad_nombre}. Se omite simulación.")
            return

        p.dict_capacidades[hospital_id][id_unidad] = nueva
        print(f"📌 Capacidad modificada en parámetros: H{hospital_id} {unidad_nombre} = {nueva}")

    for i in range(num_simulaciones):
        seed = seed_inicial + i
        print(f"\n⏳ Simulación {i+1}/{num_simulaciones} con seed={seed}")
        Paciente.CONTADOR_ID = 1

        # Crear modelo y simulación
        modelo = clase_modelo()
        simu = Simulacion(
            T_max, seed, ciclos,
            modelo=modelo,
            pacientes_caso_base=pacientes_caso_base,
            log_detallado=log_detallado
        )

        # Simular
        df = simu.simular()

        # Calcular y guardar KPIs
        t0 = time.time()
        kpis = calcular_kpis(
            df,
            save_plot=True,
            modelo=modelo,
            seed=seed,
            ciclos=ciclos,
            save_dir=plots_dir
        )
        print(f"✅ KPIs calculados en {time.time() - t0:.2f} segundos")

        # Guardar log CSV
        ruta_csv = os.path.join(logs_dir, f"{seed}.csv")
        df.to_csv(ruta_csv, index=False)
        print(f"📄 Log guardado en: {ruta_csv}")

        # Guardar KPIs JSON
        ruta_json = os.path.join(kpis_dir, f"{seed}.json")
        with open(ruta_json, 'w') as f:
            json.dump(kpis, f, indent=4)
        print(f"📊 KPIs guardados en: {ruta_json}")

    # Restaurar valor original en parámetros
    if info_cambio:
        p.dict_capacidades[hospital_id][id_unidad] = original
        print(f"🔄 Capacidad restaurada en parámetros: H{hospital_id} {unidad_nombre} = {original}")

In [ ]:
"""
Unidades: OR, ICU, SDU/WARD, GA, ED
Hospitales: 1, 2, 3
"""

correr_simulaciones_sensibilidad_simple(
    clase_modelo=ModeloA,  # No necesitas modificar el modelo
    info_cambio={"unidad": "SDU/WARD", "hospital": 3, "delta": 100},
    num_simulaciones=1,
    seed_inicial=1
)